In [2]:
import numpy as np
from scipy.linalg import sqrtm

class AmbiguityAnalyzer:
    def __init__(self, channel_func, dim=2):
        self.channel = channel_func
        self.dim = dim
    
    def trace_distance(self, rho1, rho2):
        """Compute trace distance between two density matrices."""
        diff = rho1 - rho2
        eigvals = np.linalg.eigvalsh(diff)
        return 0.5 * np.sum(np.abs(eigvals))
    
    def compute_ambiguity_matrix(self, states, epsilon=1e-6):
        """
        Compute which states become ambiguous after the channel.
        
        Parameters:
        states: List of input density matrices
        epsilon: Tolerance for considering states as ambiguous
        
        Returns:
        ambiguity_matrix: Binary matrix where A[i,j] = 1 if states i and j
                          become indistinguishable after the channel
        """
        N = len(states)
        ambiguity_matrix = np.zeros((N, N), dtype=int)
        
        # Apply channel to all states
        channel_states = [self.channel(state) for state in states]
        
        # Check pairwise ambiguity
        for i in range(N):
            for j in range(i+1, N):
                distance = self.trace_distance(channel_states[i], channel_states[j])
                if distance < epsilon:
                    ambiguity_matrix[i, j] = 1
                    ambiguity_matrix[j, i] = 1
        
        return ambiguity_matrix
    
    def ambiguity_measure(self, states, epsilon=1e-6):
        """
        Compute quantitative measure of ambiguity.
        
        Returns:
        ambiguity_score: Fraction of state pairs that become ambiguous
        avg_ambiguity_set_size: Average size of ambiguity sets
        """
        N = len(states)
        A = self.compute_ambiguity_matrix(states, epsilon)
        
        # Fraction of ambiguous pairs
        ambiguous_pairs = np.sum(A) / 2  # Divide by 2 because matrix is symmetric
        total_pairs = N * (N - 1) / 2
        ambiguity_score = ambiguous_pairs / total_pairs if total_pairs > 0 else 0
        
        # Average ambiguity set size
        ambiguity_sets = []
        for i in range(N):
            set_size = np.sum(A[i, :])
            if set_size > 0:  # Only count states with ambiguities
                ambiguity_sets.append(set_size)
        
        avg_ambiguity_set_size = np.mean(ambiguity_sets) if ambiguity_sets else 0
        
        return ambiguity_score, avg_ambiguity_set_size, A
    
    def find_ambiguous_pairs(self, states, epsilon=1e-6):
        """Find specific pairs of states that become ambiguous."""
        A = self.compute_ambiguity_matrix(states, epsilon)
        pairs = []
        
        for i in range(len(states)):
            for j in range(i+1, len(states)):
                if A[i, j] == 1:
                    pairs.append((i, j))
        
        return pairs
    
    def analyze_channel_ambiguity(self, num_samples=100, epsilon=1e-6):
        """
        Comprehensive analysis of channel ambiguity.
        
        Parameters:
        num_samples: Number of random states to sample
        epsilon: Ambiguity tolerance
        
        Returns:
        Dictionary with ambiguity statistics
        """
        # Generate random input states
        states = []
        for _ in range(num_samples):
            # Generate random pure state
            vec = np.random.randn(self.dim) + 1j * np.random.randn(self.dim)
            vec = vec / np.linalg.norm(vec)
            state = np.outer(vec, vec.conj())
            states.append(state)
        
        # Compute ambiguity
        ambiguity_score, avg_set_size, A = self.ambiguity_measure(states, epsilon)
        ambiguous_pairs = self.find_ambiguous_pairs(states, epsilon)
        
        # Additional analysis: channel contraction
        contraction_factors = []
        for i in range(min(50, num_samples)):
            for j in range(i+1, min(50, num_samples)):
                input_dist = self.trace_distance(states[i], states[j])
                output_dist = self.trace_distance(self.channel(states[i]), 
                                                  self.channel(states[j]))
                if input_dist > 1e-10:
                    contraction = output_dist / input_dist
                    contraction_factors.append(contraction)
        
        avg_contraction = np.mean(contraction_factors) if contraction_factors else 1
        
        return {
            'ambiguity_score': ambiguity_score,
            'avg_ambiguity_set_size': avg_set_size,
            'num_ambiguous_pairs': len(ambiguous_pairs),
            'avg_contraction': avg_contraction,
            'ambiguity_matrix': A,
            'ambiguous_pairs': ambiguous_pairs
        }

# Example: Fiber noise channel
def fiber_noise_channel(rho, gamma=0.1, lambda_val=0.05):
    """Combined amplitude and phase damping."""
    # Amplitude damping
    K0_ad = np.array([[1, 0], [0, np.sqrt(1-gamma)]], dtype=complex)
    K1_ad = np.array([[0, np.sqrt(gamma)], [0, 0]], dtype=complex)
    
    # Phase damping
    K0_pd = np.array([[1, 0], [0, np.sqrt(1-lambda_val)]], dtype=complex)
    K1_pd = np.array([[0, 0], [0, np.sqrt(lambda_val)]], dtype=complex)
    
    rho_ad = K0_ad @ rho @ K0_ad.conj().T + K1_ad @ rho @ K1_ad.conj().T
    rho_total = K0_pd @ rho_ad @ K0_pd.conj().T + K1_pd @ rho_ad @ K1_pd.conj().T
    
    return (rho_total + rho_total.conj().T) / 2  # Ensure Hermitian

# Analyze ambiguity for different noise levels
print("Ambiguity Analysis for Fiber Noise Channel")
print("=" * 50)

noise_levels = [(0.0, 0.0), (0.05, 0.05), (0.1, 0.1), (0.2, 0.2), (0.3, 0.3)]

for gamma, lambda_val in noise_levels:
    channel = lambda rho: fiber_noise_channel(rho, gamma, lambda_val)
    analyzer = AmbiguityAnalyzer(channel, dim=2)
    
    results = analyzer.analyze_channel_ambiguity(num_samples=100, epsilon=1e-4)
    
    print(f"\nNoise: γ={gamma}, λ={lambda_val}")
    print(f"  Ambiguity score: {results['ambiguity_score']:.4f}")
    print(f"  Avg ambiguity set size: {results['avg_ambiguity_set_size']:.2f}")
    print(f"  Number of ambiguous pairs: {results['num_ambiguous_pairs']}")
    print(f"  Avg contraction factor: {results['avg_contraction']:.4f}")
    
    # Show example of ambiguous states
    if results['ambiguous_pairs']:
        i, j = results['ambiguous_pairs'][0]
        print(f"  Example ambiguous pair: states {i} and {j}")

Ambiguity Analysis for Fiber Noise Channel

Noise: γ=0.0, λ=0.0
  Ambiguity score: 0.0000
  Avg ambiguity set size: 0.00
  Number of ambiguous pairs: 0
  Avg contraction factor: 1.0000

Noise: γ=0.05, λ=0.05
  Ambiguity score: 0.0000
  Avg ambiguity set size: 0.00
  Number of ambiguous pairs: 0
  Avg contraction factor: 0.9500

Noise: γ=0.1, λ=0.1
  Ambiguity score: 0.0000
  Avg ambiguity set size: 0.00
  Number of ambiguous pairs: 0
  Avg contraction factor: 0.9000

Noise: γ=0.2, λ=0.2
  Ambiguity score: 0.0000
  Avg ambiguity set size: 0.00
  Number of ambiguous pairs: 0
  Avg contraction factor: 0.8000

Noise: γ=0.3, λ=0.3
  Ambiguity score: 0.0000
  Avg ambiguity set size: 0.00
  Number of ambiguous pairs: 0
  Avg contraction factor: 0.7000
